In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [3]:
#read the dataset
data = pd.read_csv('data-set/Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
#preprocess of data
#drop irrelevant columns

data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
#encode categorcial data
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
# data

In [6]:
#OneHOt encoding to Geography col
from sklearn.preprocessing import OneHotEncoder
onehot_encoder = OneHotEncoder(sparse_output=False)
geo_encoder = onehot_encoder.fit_transform(data[['Geography']])
geo_encoder

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [7]:
#get values of Geography as feature names
onehot_encoder.get_feature_names_out(['Geography'])

#convert encoded value to dataframe
encoded_geo_df = pd.DataFrame(geo_encoder, columns=onehot_encoder.get_feature_names_out(['Geography']))

encoded_geo_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [8]:
#combine column for geography to original dataset
data = pd.concat([data.drop('Geography', axis=1), encoded_geo_df], axis=1 )

data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [9]:
# save encoder/scaler into pickle file
import pickle
with open('pickles/labelencoder.pkl', 'wb') as f:
    pickle.dump(label_encoder_gender, f)

with open('pickles/onehotencoder.pkl', 'wb') as f:
    pickle.dump(onehot_encoder, f)

In [10]:
# divide data into dependent and independent features

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#scale these features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(x_test)

In [11]:
#save scaler into pickle
with open('pickles/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

## ANN Implementation

In [12]:
#create sequential network, dense
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
import datetime

In [13]:
X_train.shape[1]

12

In [14]:
data.shape

(10000, 13)

In [15]:
model = Sequential([
    Input(shape=(X_train.shape[1], )),
    Dense(64, activation='relu'), #first Hidden Layer connected with input layer
    Dense(32, activation='relu'), #HL2 layer
    Dense(1, activation='sigmoid'), #Output layer
])

In [16]:
model

<Sequential name=sequential, built=True>

In [17]:
#apply backward and forward propagation, after compile of model
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate=0.001) #get the optimizer
loss = tensorflow.keras.losses.BinaryCrossentropy() #select loss method

In [18]:
##compile the model
model.compile(optimizer=opt, loss=loss, metrics=['accuracy'])

In [19]:
## Setup tensorboard
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
# log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
#
# tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=0)

In [20]:
#setup early stopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [21]:
#training of the model
# history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, callbacks=[tensorboard_callback, early_stopping_callback], batch_size=512, verbose=2)

In [22]:
# model.save('models/model.h5')

In [28]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


## Prediction from the model

In [29]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 27931), started 0:03:08 ago. (Use '!kill 27931' to kill it.)